In [1]:
!uv pip install spacy

Resolved 46 packages in 989ms
Installed 30 packages in 775ms
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + blis==1.3.3
 + catalogue==2.0.10
 + click==8.5.0
 + cloudpathlib==0.25.0
 + confection==1.3.3
 + cymem==2.0.13
 + markdown-it-py==4.2.0
 + mdurl==0.1.2
 + murmurhash==1.0.15
 + numpy==2.2.6
 + preshed==3.0.13
 + pydantic==2.13.5
 + pydantic-core==2.46.5
 + rich==15.0.0
 + setuptools==84.0.0
 + shellingham==1.5.4
 + smart-open==8.0.1
 + spacy==3.8.16
 + spacy-legacy==3.0.12
 + spacy-loggers==1.0.5
 + srsly==2.5.3
 + thinc==8.3.13
 + tqdm==4.70.1
 + typer==0.27.2
 + typing-inspection==0.4.4
 + wasabi==1.1.3
 + weasel==1.0.0
 + wrapt==2.4.1


In [2]:
import re
import spacy
from spacy.matcher import Matcher

In [3]:
!python -m spacy download en_core_web_sm

[+] Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


Resolved 1 package in 21.89s
Installed 1 package in 36ms
 + en-core-web-sm==3.8.0 (from https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl)


In [4]:
nlp = spacy.load("en_core_web_sm")
ruler = nlp.add_pipe("entity_ruler", before="ner", config={"overwrite_ents": True})

In [5]:
CONFIDENCE_THRESHOLD = 0.75

In [6]:
patterns = [
    {"label": "BRAND", "pattern": [{"LOWER": "coach"}], "id": "coach"},
 
    {"label": "BRAND", "pattern": [{"LOWER": "nike"}], "id": "nike"},
    {"label": "BRAND", "pattern": [{"LOWER": "niike"}], "id": "nike"},
    {"label": "BRAND", "pattern": [{"LOWER": "nikee"}], "id": "nike"},
 
    {"label": "BRAND", "pattern": [{"LOWER": "adidas"}], "id": "adidas"},
    {"label": "BRAND", "pattern": [{"LOWER": "addidas"}], "id": "adidas"},
    {"label": "BRAND", "pattern": [{"LOWER": "adiddas"}], "id": "adidas"},
 
    {"label": "BRAND", "pattern": [{"LOWER": "gap"}], "id": "gap"},
    {"label": "BRAND", "pattern": [{"LOWER": "guess"}], "id": "guess"},
    {"label": "BRAND", "pattern": [{"LOWER": "target"}], "id": "target"},
 
    {"label": "BRAND", "pattern": [{"LOWER": "h&m"}], "id": "h&m"},
    {"label": "BRAND", "pattern": [{"LOWER": "h"}, {"LOWER": "and"}, {"LOWER": "m"}], "id": "h&m"},
    {"label": "BRAND", "pattern": [{"LOWER": "h"}, {"ORTH": "&"}, {"LOWER": "m"}], "id": "h&m"},
 
    {"label": "CATEGORY", "pattern": [{"LOWER": "bag"}], "id": "bags"},
    {"label": "CATEGORY", "pattern": [{"LOWER": "bags"}], "id": "bags"},
    {"label": "CATEGORY", "pattern": [{"LOWER": "handbag"}], "id": "bags"},
    {"label": "CATEGORY", "pattern": [{"LOWER": "handbags"}], "id": "bags"},
 
    {"label": "CATEGORY", "pattern": [{"LOWER": "shoe"}], "id": "shoes"},
    {"label": "CATEGORY", "pattern": [{"LOWER": "shoes"}], "id": "shoes"},
    {"label": "CATEGORY", "pattern": [{"LOWER": "sneaker"}], "id": "shoes"},
    {"label": "CATEGORY", "pattern": [{"LOWER": "sneakers"}], "id": "shoes"},
 
    {"label": "CATEGORY", "pattern": [{"LOWER": "watch"}], "id": "watches"},
    {"label": "CATEGORY", "pattern": [{"LOWER": "watches"}], "id": "watches"},
 
    {"label": "CATEGORY", "pattern": [{"LOWER": "dress"}], "id": "dresses"},
    {"label": "CATEGORY", "pattern": [{"LOWER": "dresses"}], "id": "dresses"},
    {"label": "CATEGORY", "pattern": [{"LOWER": "gown"}], "id": "dresses"},
    {"label": "CATEGORY", "pattern": [{"LOWER": "gowns"}], "id": "dresses"},
 
    {"label": "CATEGORY", "pattern": [{"LOWER": "jacket"}], "id": "jackets"},
    {"label": "CATEGORY", "pattern": [{"LOWER": "jackets"}], "id": "jackets"},
    {"label": "CATEGORY", "pattern": [{"LOWER": "coat"}], "id": "jackets"},
    {"label": "CATEGORY", "pattern": [{"LOWER": "coats"}], "id": "jackets"},
 
    {"label": "GENDER", "pattern": [{"LOWER": "men"}], "id": "men"},
    {"label": "GENDER", "pattern": [{"LOWER": "mens"}], "id": "men"},
    {"label": "GENDER", "pattern": [{"LOWER": "men's"}], "id": "men"},
    {"label": "GENDER", "pattern": [{"LOWER": "man"}], "id": "men"},
 
    {"label": "GENDER", "pattern": [{"LOWER": "women"}], "id": "women"},
    {"label": "GENDER", "pattern": [{"LOWER": "womens"}], "id": "women"},
    {"label": "GENDER", "pattern": [{"LOWER": "women's"}], "id": "women"},
    {"label": "GENDER", "pattern": [{"LOWER": "woman"}], "id": "women"},
    {"label": "GENDER", "pattern": [{"LOWER": "ladies"}], "id": "women"},
 
    {"label": "GENDER", "pattern": [{"LOWER": "kids"}], "id": "kids"},
    {"label": "GENDER", "pattern": [{"LOWER": "kid's"}], "id": "kids"},
    {"label": "GENDER", "pattern": [{"LOWER": "children"}], "id": "kids"},
    {"label": "GENDER", "pattern": [{"LOWER": "boys"}], "id": "kids"},
    {"label": "GENDER", "pattern": [{"LOWER": "girls"}], "id": "kids"},
 
    {"label": "GENDER", "pattern": [{"LOWER": "unisex"}], "id": "unisex"},
]

In [7]:
color_patterns = [
    {"label": "COLOR", "pattern": [{"LOWER": c}], "id": canonical}
    for canonical, variants in {
        "black": ["black"],
        "white": ["white"],
        "red": ["red"],
        "blue": ["blue", "navy"],
        "green": ["green", "olive"],
        "orange": ["orange"],
        "pink": ["pink"],
        "grey": ["grey", "gray", "charcoal"],
    }.items()
    for c in variants
]

In [8]:
ruler.add_patterns(patterns + color_patterns)

In [9]:
price_matcher = Matcher(nlp.vocab)

In [13]:
CURRENCY_WORDS = ["dollars", "usd", "bucks", "$"]

In [14]:
price_matcher.add("PRICE_LT", [
    [{"LOWER": {"IN": ["under", "below"]}}, {"LOWER": "$"}, {"LIKE_NUM": True}],
    [{"LOWER": {"IN": ["under", "below"]}}, {"LIKE_NUM": True}, {"LOWER": {"IN": CURRENCY_WORDS}}],
    [{"LOWER": "less"}, {"LOWER": "than"}, {"LOWER": "$"}, {"LIKE_NUM": True}],
    [{"LOWER": "less"}, {"LOWER": "than"}, {"LIKE_NUM": True}, {"LOWER": {"IN": CURRENCY_WORDS}}],
])
 
price_matcher.add("PRICE_GT", [
    [{"LOWER": {"IN": ["over", "above"]}}, {"LOWER": "$"}, {"LIKE_NUM": True}],
    [{"LOWER": {"IN": ["over", "above"]}}, {"LIKE_NUM": True}, {"LOWER": {"IN": CURRENCY_WORDS}}],
    [{"LOWER": "more"}, {"LOWER": "than"}, {"LOWER": "$"}, {"LIKE_NUM": True}],
    [{"LOWER": "more"}, {"LOWER": "than"}, {"LIKE_NUM": True}, {"LOWER": {"IN": CURRENCY_WORDS}}],
])
 
price_matcher.add("PRICE_RANGE", [
    [{"LOWER": "between"}, {"LOWER": "$"}, {"LIKE_NUM": True}, {"LOWER": "and"}, {"LOWER": "$"}, {"LIKE_NUM": True}],
    [{"LOWER": "between"}, {"LIKE_NUM": True}, {"LOWER": "and"}, {"LIKE_NUM": True}, {"LOWER": {"IN": CURRENCY_WORDS}}],
])
 
price_matcher.add("PRICE_EXACT", [
    [{"TEXT": {"REGEX": r"^\$\d+(\.\d+)?$"}}],
    [{"LIKE_NUM": True}, {"LOWER": {"IN": CURRENCY_WORDS}}],
])

In [15]:
def extract_price(doc) -> list[dict]:
    matches = price_matcher(doc)
    results, seen = [], set()
 
    for match_id, start, end in matches:
        if any(i in seen for i in range(start, end)):
            continue
        seen.update(range(start, end))
 
        label = nlp.vocab.strings[match_id]
        span = doc[start:end]
        nums = [float(t.text.replace("$", "")) for t in span if t.like_num]
 
        entry = {"span": span.text, "confidence": 0.95}  # currency marker always present now
        if label == "PRICE_LT" and nums:
            results.append({**entry, "operator": "LT", "value": nums[0]})
        elif label == "PRICE_GT" and nums:
            results.append({**entry, "operator": "GT", "value": nums[0]})
        elif label == "PRICE_RANGE" and len(nums) == 2:
            results.append({**entry, "operator": "BETWEEN", "value": nums})
        elif label == "PRICE_EXACT" and nums:
            results.append({**entry, "operator": "EQ", "value": nums[0]})
 
    return results

In [24]:
def extract_vocab_facets(doc) -> list[dict]:
    facets = []
    for ent in doc.ents:
        token = ent.root
        pos = token.pos_
 
        if ent.label_ == "BRAND":
            valid = pos == "PROPN"
            # PROPN + title case = strong signal; PROPN alone (lowercase input) = weaker
            confidence = 0.9 if valid else 0.2
        else:
            valid = pos in {"NOUN", "ADJ"}
            confidence = 0.9 if valid else 0.2
 
        facets.append({
            "text": ent.text,
            "label": ent.label_,
            "value": ent.ent_id_ or ent.text.lower(),
            "valid": valid,
            "confidence": round(confidence, 2),
        })
    return [f for f in facets if f["valid"]]

In [25]:
def get_categories(facets: list[dict]) -> list[dict]:
    return [f for f in facets if f["label"] == "CATEGORY"]
 
 
def get_gender(facets: list[dict]) -> list[dict]:
    return [f for f in facets if f["label"] == "GENDER"]
 
 
def get_filters(facets: list[dict], price: list[dict]) -> dict:
    return {
        "color": [f for f in facets if f["label"] == "COLOR"],
        "brand": [f for f in facets if f["label"] == "BRAND"],
        "price": price,
    }

In [26]:
ORDINAL_WORDS = {
    "first": 1, "second": 2, "third": 3, "fourth": 4, "fifth": 5,
    "sixth": 6, "seventh": 7, "eighth": 8, "ninth": 9, "tenth": 10,
}
 
ORDINAL_SUFFIX_RE = re.compile(r"^(\d{1,2})(st|nd|rd|th)$", re.IGNORECASE)
 
# Intents where a bare number plausibly refers to a product position rather
# than a quantity/price. Only these intents allow bare-cardinal indices.
POSITION_REFERENCING_INTENTS = {"compare", "shop-the-look", "theme-browse"}
 
# Verbs/words that signal the sentence is referencing existing result items
REFERENCE_CONTEXT_WORDS = {
    "compare", "show", "like", "similar", "more", "item", "items",
    "one", "ones", "product", "products", "option", "options",
}

In [27]:
def extract_indices(doc, intent: str | None = None) -> list[dict]:
    results = []
 
    for i, token in enumerate(doc):
        # Ordinal word: "third", "fourth" — unambiguous, always high confidence
        if token.lower_ in ORDINAL_WORDS:
            results.append({
                "index": ORDINAL_WORDS[token.lower_],
                "text": token.text,
                "confidence": 0.95,
            })
            continue
 
        # Numeric ordinal: "3rd", "4th" — unambiguous, always high confidence
        m = ORDINAL_SUFFIX_RE.match(token.text)
        if m:
            results.append({
                "index": int(m.group(1)),
                "text": token.text,
                "confidence": 0.95,
            })
            continue
 
        # Bare cardinal number: "compare 3 and 4" — ambiguous with
        # price/quantity, only accept with supporting context.
        if token.like_num and token.pos_ == "NUM":
            window = doc[max(0, i - 3):min(len(doc), i + 3)]
            has_reference_context = any(t.lower_ in REFERENCE_CONTEXT_WORDS for t in window)
            intent_supports_position = intent in POSITION_REFERENCING_INTENTS
 
            if has_reference_context or intent_supports_position:
                try:
                    idx = int(float(token.text))
                except ValueError:
                    continue
                confidence = 0.85 if (has_reference_context and intent_supports_position) else 0.6
                results.append({"index": idx, "text": token.text, "confidence": confidence})
 
    # de-duplicate same index, keep highest-confidence mention
    best_by_index = {}
    for r in results:
        if r["index"] not in best_by_index or r["confidence"] > best_by_index[r["index"]]["confidence"]:
            best_by_index[r["index"]] = r
 
    return sorted(best_by_index.values(), key=lambda r: r["index"])

In [28]:
def extract(query: str, intent: str | None = None) -> dict:
    doc = nlp(query)
 
    vocab_facets = extract_vocab_facets(doc)
    price = extract_price(doc)
    indices = extract_indices(doc, intent=intent)
 
    all_confidences = (
        [f["confidence"] for f in vocab_facets]
        + [p["confidence"] for p in price]
        + [idx["confidence"] for idx in indices]
    )
    # No facets extracted at all is its own signal — don't default to 1.0
    overall_confidence = min(all_confidences) if all_confidences else 0.5
 
    return {
        "query": query,
        "intent": intent,
        "categories": get_categories(vocab_facets),
        "gender": get_gender(vocab_facets),
        "filters": get_filters(vocab_facets, price),
        "indices": [r["index"] for r in indices],
        "indices_detail": indices,
        "overall_confidence": round(overall_confidence, 2),
        "needs_llm": overall_confidence < CONFIDENCE_THRESHOLD,
    }

In [35]:
extract(input("Query: "), input("Intent: "))

Query:  compare 1 and 2
Intent:  compare


{'query': 'compare 1 and 2',
 'intent': 'compare',
 'categories': [],
 'gender': [],
 'filters': {'color': [], 'brand': [], 'price': []},
 'indices': [1, 2],
 'indices_detail': [{'index': 1, 'text': '1', 'confidence': 0.85},
  {'index': 2, 'text': '2', 'confidence': 0.85}],
 'overall_confidence': 0.85,
 'needs_llm': False}